In [1]:
print("a")

a


In [2]:
from pathlib import Path
import requests

from pystac_client import Client
import planetary_computer

OUT_DIR = Path("../datasets/sentinel2_visual")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 例: 東京湾のざっくり bbox [min_lon, min_lat, max_lon, max_lat]
bbox = [139.5, 35.2, 140.2, 35.9]

catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2024-01-01/2025-12-31",
    query={"eo:cloud_cover": {"lt": 20}},
)

items = list(search.items())

print(f"found: {len(items)} items")

for item in items[:100]:
    asset = item.assets.get("visual")
    if asset is None:
        continue

    url = asset.href
    out = OUT_DIR / f"{item.id}_visual.tif"
    if out.exists():
        continue

    print("downloading", out.name)
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(out, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

In [ ]:
import time

# 既存の OUT_DIR / requests / Client / planetary_computer を利用
# 必要なら catalog を再作成（このセル単体実行でも動くように）
if "catalog" not in globals():
    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace,
    )

regions: dict[str, list[float]] = {
    # Asia-Pacific
    "tokyo_bay_jp": [139.5, 35.2, 140.2, 35.9],
    "osaka_bay_jp": [134.9, 34.3, 135.6, 34.9],
    "singapore_strait_sg": [103.4, 1.0, 104.3, 1.6],
    "manila_bay_ph": [120.6, 14.2, 121.1, 14.9],
    "jakarta_bay_id": [106.5, -6.3, 107.2, -5.8],
    "sydney_au": [150.8, -34.2, 151.5, -33.5],
    "auckland_nz": [174.4, -37.2, 175.1, -36.6],

    # Europe
    "north_sea_nl": [3.2, 51.5, 5.2, 53.0],
    "english_channel_uk_fr": [-2.5, 49.5, 1.5, 51.2],
    "mediterranean_it": [12.0, 40.0, 15.0, 42.0],
    "aegean_gr": [23.0, 36.5, 26.5, 39.5],
    "norwegian_coast_no": [4.0, 58.0, 8.5, 62.0],

    # Africa / Middle East
    "gulf_of_guinea_ng": [2.5, 4.0, 8.5, 6.8],
    "cape_town_za": [17.8, -34.5, 19.2, -33.3],
    "red_sea_eg_sa": [34.0, 21.0, 39.5, 27.5],
    "persian_gulf_ae_ir": [52.0, 24.0, 56.5, 27.8],

    # Americas
    "new_york_us": [-74.4, 40.3, -73.4, 41.1],
    "san_francisco_us": [-123.0, 37.0, -121.5, 38.3],
    "gulf_of_mexico_us": [-91.5, 27.0, -88.0, 30.0],
    "caribbean_pa": [-80.5, 8.4, -78.5, 10.2],
    "rio_brazil_br": [-44.0, -23.6, -42.8, -22.6],
    "chile_coast_cl": [-73.5, -34.5, -71.0, -32.0],

    # Open ocean (coastline以外も確保)
    "north_pacific_open_ocean": [-160.0, 20.0, -150.0, 30.0],
    "south_indian_open_ocean": [70.0, -35.0, 85.0, -25.0],
}

date_range = "2024-01-01/2025-12-31"
cloud_lt = 20 # クラウドカバー率の上限（%）
max_items_per_region = 2 # 各リージョンから最大何アイテムまでダウンロードするか（Pass数）
sleep_sec = 30  # ダウンロード間のウェイト（お行儀よく）

existing = {p.name for p in OUT_DIR.glob("*_visual.tif")}
downloaded = 0
skipped = 0

# --- Phase 1: 全リージョンの候補アイテムを収集 ---
region_queues: dict[str, list] = {}
for region_name, bbox in regions.items():
    print(f"[{region_name}] searching...")
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=date_range,
        query={"eo:cloud_cover": {"lt": cloud_lt}},
    )
    items = [item for item in search.items() if item.assets.get("visual") is not None]
    region_queues[region_name] = items
    print(f"  found: {len(items)} items")

# --- Phase 2: ラウンドロビンでダウンロード（リージョン網羅を優先）---
for pass_idx in range(max_items_per_region):
    for region_name, items in region_queues.items():
        if pass_idx >= len(items):
            continue

        item = items[pass_idx]
        asset = item.assets["visual"]
        out = OUT_DIR / f"{item.id}_visual.tif"

        if out.name in existing or out.exists():
            skipped += 1
            continue

        try:
            with requests.get(asset.href, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(out, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            existing.add(out.name)
            downloaded += 1
            print(f"  [{region_name}] pass {pass_idx + 1}/{max_items_per_region}: {out.name}")
            time.sleep(sleep_sec)
        except Exception as e:
            print(f"  [{region_name}] failed: {item.id} ({e})")

print(f"\nDone. downloaded={downloaded}, skipped={skipped}, total_files={len(existing)}")
